In [1]:
import numpy as np

class AntColonyOptimizationTSP:
    def __init__(self, city_coords, n_ants=20, n_iterations=100, alpha=1.0, beta=2.0, rho=0.5, q=1.0):
        self.coords = city_coords
        self.n_cities = len(city_coords)
        self.n_ants = n_ants
        self.n_iterations = n_iterations
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.q = q


        self.distances = np.sqrt(
            ((self.coords[:, np.newaxis, :] - self.coords[np.newaxis, :, :]) ** 2).sum(axis=-1)
        )

        np.fill_diagonal(self.distances, np.inf)


        self.eta = 1.0 / self.distances


        self.pheromone = np.ones((self.n_cities, self.n_cities)) * 0.1

    def solve(self):
        best_distance = float('inf')
        best_route = None

        for iteration in range(self.n_iterations):
            all_routes = []
            all_distances = []

            for ant in range(self.n_ants):
                route = self._construct_solution()
                distance = self._calculate_route_distance(route)

                all_routes.append(route)
                all_distances.append(distance)

                if distance < best_distance:
                    best_distance = distance
                    best_route = route


            self._update_pheromones(all_routes, all_distances)


            if (iteration + 1) % 10 == 0 or iteration == 0:
                print(f"Iteration {iteration + 1}/{self.n_iterations} | Best Distance: {best_distance:.2f}")

        return best_route, best_distance

    def _construct_solution(self):

        route = [0]
        unvisited = set(range(1, self.n_cities))

        while unvisited:
            current_city = route[-1]
            unvisited_list = list(unvisited)


            pheromone_values = self.pheromone[current_city, unvisited_list] ** self.alpha
            heuristic_values = self.eta[current_city, unvisited_list] ** self.beta
            probabilities = pheromone_values * heuristic_values


            prob_sum = np.sum(probabilities)
            if prob_sum == 0:
                probabilities = np.ones_like(probabilities) / len(probabilities)
            else:
                probabilities /= prob_sum

            next_city = np.random.choice(unvisited_list, p=probabilities)
            route.append(next_city)
            unvisited.remove(next_city)


        route.append(0)
        return route

    def _calculate_route_distance(self, route):
        distance = 0.0
        for i in range(len(route) - 1):
            distance += self.distances[route[i], route[i+1]]
        return distance

    def _update_pheromones(self, all_routes, all_distances):

        self.pheromone *= (1.0 - self.rho)


        for route, distance in zip(all_routes, all_distances):
            pheromone_to_add = self.q / distance
            for i in range(len(route) - 1):
                self.pheromone[route[i], route[i+1]] += pheromone_to_add
                self.pheromone[route[i+1], route[i]] += pheromone_to_add


if __name__ == "__main__":

    np.random.seed(42)


    num_cities = 15
    delivery_locations = np.random.rand(num_cities, 2) * 100

    print("Initializing ACO for delivery route optimization...")
    aco = AntColonyOptimizationTSP(
        city_coords=delivery_locations,
        n_ants=15,
        n_iterations=100,
        alpha=1.0,
        beta=2.5,
        rho=0.4
    )

    best_path, min_dist = aco.solve()

    print("\n--- Optimization Complete ---")
    print(f"Shortest Route found: {' -> '.join(map(str, best_path))}")
    print(f"Minimum Distance required: {min_dist:.2f} units")


Initializing ACO for delivery route optimization...
Iteration 1/100 | Best Distance: 426.06
Iteration 10/100 | Best Distance: 327.27
Iteration 20/100 | Best Distance: 320.53
Iteration 30/100 | Best Distance: 320.53
Iteration 40/100 | Best Distance: 320.53
Iteration 50/100 | Best Distance: 320.53
Iteration 60/100 | Best Distance: 320.53
Iteration 70/100 | Best Distance: 320.53
Iteration 80/100 | Best Distance: 320.53
Iteration 90/100 | Best Distance: 320.53
Iteration 100/100 | Best Distance: 320.53

--- Optimization Complete ---
Shortest Route found: 0 -> 5 -> 3 -> 13 -> 8 -> 11 -> 7 -> 2 -> 9 -> 14 -> 10 -> 6 -> 1 -> 4 -> 12 -> 0
Minimum Distance required: 320.53 units
